In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#importing necessary files
merged_df = pd.read_csv('merged_df.tsv', sep='\t')
plot_summary_df = pd.read_csv('plot_summaries.txt', sep='\t')

In [ ]:
def estimate_release_year(row, mean_release_year_by_genre):
    """
    Function to estimate the release year of a movie based on its genres (by taking the average of the release years of movies with the same genres)

    Parameters: row (pd.Series) - a row of a pandas DataFrame

    Returns: release_year (int) - the estimated release year of the movie
    """
    if pd.isna(row['Movie_release_date']):
        genres = row['genre_list']
        mean_years = [mean_release_year_by_genre[genre] for genre in genres if genre in mean_release_year_by_genre]
        if mean_years:
            return sum(mean_years) / len(mean_years)
        else:
            return pd.NA
    else:
        return row['Movie_release_date']

In [ ]:
#Estimation of the release year of the films without release data
mean_release_year_by_genre = pd.read_csv('mean_release_year_by_genre.tsv', sep='\t')
merged_df['Estimated_release_year'] = merged_df.apply(estimate_release_year, axis=1, args=(mean_release_year_by_genre,))
merged_df['Estimated_release_year'] = pd.to_datetime(merged_df['Estimated_release_year'], errors='coerce').dt.year
merged_df['Decade'] = (merged_df['Estimated_release_year'] // 10) * 10

In [ ]:
#initializing decades 
decades=np.arange(1900,2020,10).astype(float)

# extracting a list of the movie ids for each decade
movies_ids_per_decade={} # dict of the lists of movie ids for all decades
for decade in decades:
    movies_df=merged_df[merged_df['Decade']==decade]
    decade_ids=movies_df['Wikipedia_movie_ID'].to_list()
    decade_ids=list(dict.fromkeys(decade_ids))
    movies_ids_per_decade[decade]=decade_ids

# creating a list of the summaries of the decade for each decade
summaries_per_decade={} # dict of the lists of movie summaries for all decades
for decade in decades :
    summaries_list=[]
    for id in movies_ids_per_decade[decade] :
        summary=plot_summary_df[plot_summary_df['movie_id']==id]['plot_summary'].to_list() # extracting the plot summaries from the movie ids in movies_ids_per_decade
        if(summary!=[]):
            summaries_list.append(summary[0])
    summaries_per_decade[decade]=summaries_list

In [ ]:
with open('summaries_per_decade.txt', 'w') as fichier:
    for key, valeur in summaries_per_decade.items():
        fichier.write(f'{key}: {valeur}\n')